# TweetAge v4 – Homofilia ampliada, diversidad de modelos base y pesos por clase
**Curso de Procesamiento del Lenguaje Natural · Maestría en Inteligencia Artificial · Pontificia Universidad Javeriana**

Equipo: ____________________

Sin modelos preentrenados: todo se aprende desde cero con `age_train.csv`. La arquitectura tipo fastText queda para un notebook posterior.

## Evolución de los resultados

| Versión | Modelo | F1 macro validación (80/20) | F1 macro Kaggle |
|---|---|---|---|
| v1 | SVM lineal sobre TF-IDF con vocabulario limitado | 0,555 | 0,56175 |
| v2 | NB-SVM sobre palabras 1-3 + caracteres 2-5 y 1-5 | 0,581 | — |
| v3 | NB-SVM + homofilia de red (G6) + meta-modelo de gradient boosting | 0,607 | 0,61765 |
| **v4** | **v3 + homofilia ampliada + modelos base diversos + estilo + pesos por clase** | **0,619** | por medir |

## Qué cambia respecto a la v3
Cada cambio se midió en la **misma partición 80/20 (semilla 42)** y con las mismas particiones fuera de muestra de la v3:

| Paso | Entradas del meta-modelo | F1 macro | Cambio |
|---|---|---|---|
| v3 | NB-SVM + G6 (homofilia @cuentas y #hashtags) | 0,6074 | — |
| 1 | + **H5: homofilia ampliada** (cinco tipos de relación por separado) | 0,6111 | +0,004 |
| 2 | + **modelos base diversos** (NB-SVM solo palabras, ComplementNB) | 0,6127 | +0,002 |
| 3 | + **G0: estilo y legibilidad** (ahora con meta-modelo no lineal) | 0,6182 | +0,006 |
| 4 | + **pesos por clase** ajustados para el F1 macro | **0,6193** | +0,001 a +0,003 |

**Se probaron y se descartaron** (sin ganancia o con un costo de tiempo que no se justifica):

| Experimento | Resultado |
|---|---|
| NB-SVM solo de caracteres como modelo base extra | sin ganancia y +8 min por fase |
| Clasificadores ordinales acumulativos (Frank & Hall, 2001) | +0,0004 y +9 min por fase |
| Regresión ordinal Ridge (G7) | +0,0004 |
| Ciclo vital, psicológicas, jerga, morfología, complejidad (G1-G5) | +0,0002 |
| Temas NMF (G8) | −0,0002 |
| Propagación en el grafo de cuentas | +0,0003 frente a H5 sin propagación |
| Ajuste fino del gradient boosting (tasa de aprendizaje, iteraciones) | ±0,001 |

> **Por qué el estilo (G0) entra ahora y no en la v3:** el selector de la v3 comparaba familias con una regresión logística, que no capta interacciones. Con el meta-modelo de gradient boosting, las variables de estilo ayudan a desempatar casos en los que NB-SVM duda entre grupos vecinos.

**Tiempo estimado:** 35-45 min con `VALIDAR=True`; 18-25 min con `VALIDAR=False`. Requiere unos 8 GB de RAM.

## 1. Librerías y configuración

Solo `numpy`, `pandas`, `scipy`, `scikit-learn` y `matplotlib`.

| Parámetro | Valor | Significado |
|---|---|---|
| `VALIDAR` | `True` | Repite todo en un holdout 80/20 y muestra la tabla de ablación, métricas por clase e importancias antes del modelo final |
| `SAMPLE_FRAC` | `None` | Fracción del train para probar el código rápido (p. ej. 0,1); no sirve para subir a Kaggle |
| `K_FOLDS` | 3 | Particiones para las variables fuera de muestra |
| `NB_ALPHA`, `SVM_C` | 1,0 y 0,3 | Hiperparámetros de NB-SVM elegidos en la v2 |
| `HGB_PARAMS` | ver celda | Meta-modelo de gradient boosting (hojas más grandes que en la v3) |

In [ ]:
import os, re, html, time, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay
warnings.filterwarnings('ignore')

VALIDAR     = True
SAMPLE_FRAC = None
SEED        = 42
K_FOLDS     = 3
NB_ALPHA, SVM_C = 1.0, 0.3
HGB_PARAMS  = dict(max_iter=300, learning_rate=0.06, max_leaf_nodes=63, min_samples_leaf=50,
                   l2_regularization=1.0, random_state=0)
OUT_PATH    = 'age_submission_v4.csv'

CANDIDATE_DIRS = [os.getcwd(), os.path.join(os.getcwd(), 'data'), r'/kaggle/input/tweet-age-2026-puj',
                  r'C:\Users\leonk\Documents\MINTA 2025-01\PLN']
DATA_DIR = next((d for d in CANDIDATE_DIRS if os.path.exists(os.path.join(d, 'age_train.csv'))), None)
assert DATA_DIR, 'No encontré age_train.csv: agregue su carpeta a CANDIDATE_DIRS.'

GROUPS = np.array(['13-17', '18-24', '25-34', '35-49', '50-64', '65-xx'])
T0 = time.time()
def log(*m): print(f'[{time.time() - T0:6.0f}s]', *m, flush=True)
log('Datos en:', DATA_DIR)

## 2. Lectura y limpieza mínima

Igual que en las versiones anteriores: separador `;`, BOM, relleno de espacios y entidades HTML decodificadas. Del entrenamiento se excluyen los tweets vacíos y el artefacto `#¿NOMBRE?`. No se eliminan stopwords, menciones, mayúsculas ni alargamientos, porque son marcas de edad.

In [ ]:
def load(path):
    df = pd.read_csv(path, sep=';', encoding='utf-8-sig', dtype=str, keep_default_na=False)
    df['text'] = df['text'].str.strip().map(html.unescape)
    return df

def is_junk(texts):
    t = pd.Series(texts)
    return ((t == '') | t.str.contains('#¿NOMBRE', regex=False)).values

train_raw = load(os.path.join(DATA_DIR, 'age_train.csv'))
test = load(os.path.join(DATA_DIR, 'age_test.csv'))
if SAMPLE_FRAC:
    train_raw = train_raw.groupby('age_range', group_keys=False).sample(frac=SAMPLE_FRAC, random_state=SEED)
train = train_raw[~is_junk(train_raw.text)].reset_index(drop=True)
log(f'train {train.shape} | test {test.shape}')

## 3. Modelos base (nivel 1)

El sistema tiene dos niveles. En el primero, varios modelos de texto producen **puntuaciones por clase**; en el segundo, un meta-modelo las combina con otras variables.

**Representación:** los tres bloques TF-IDF de la v2/v3, sin límite de vocabulario: palabras 1-3 (con `@usuarios` y `#hashtags`), caracteres 2-5 dentro de palabra (`char_wb`) y caracteres 1-5 que cruzan palabras (`char`).

**Tres modelos base, que se equivocan de forma distinta:**
1. **NB-SVM completo** (Wang & Manning, 2012) sobre los tres bloques: un SVM por clase con cada término ponderado por su log-cociente de Naive Bayes, $r_c = \log\frac{(p+\alpha)/\lVert p+\alpha\rVert_1}{(q+\alpha)/\lVert q+\alpha\rVert_1}$.
2. **NB-SVM solo de palabras** *(nuevo en v4)*: ve únicamente vocabulario, cuentas y hashtags, sin la influencia de los n-gramas de caracteres.
3. **ComplementNB** *(nuevo en v4)*: Naive Bayes que estima cada clase con los documentos de las *otras* clases, más estable en texto (Rennie et al., 2003). Es un modelo probabilístico, no de márgenes como el SVM.

Darle al meta-modelo puntuaciones de modelos distintos le permite aprender **cuándo confiar en cada uno** (*stacking*; Wolpert, 1992).

In [ ]:
def make_vectorizers():
    return [TfidfVectorizer(token_pattern=r'(?u)[@#]?\w+', ngram_range=(1, 3), min_df=2, sublinear_tf=True, dtype=np.float32),
            TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=3, lowercase=False, sublinear_tf=True, dtype=np.float32),
            TfidfVectorizer(analyzer='char', ngram_range=(1, 5), min_df=5, lowercase=False, sublinear_tf=True, dtype=np.float32)]

def nbsvm_scores(Xfit, yfit, Xapp):
    """Un SVM binario por clase sobre columnas escaladas por el log-cociente de Naive Bayes."""
    D = np.zeros((Xapp.shape[0], len(GROUPS)), np.float32)
    for i, c in enumerate(GROUPS):
        pos = yfit == c
        p = np.asarray(Xfit[pos].sum(0)).ravel() + NB_ALPHA
        q = np.asarray(Xfit[~pos].sum(0)).ravel() + NB_ALPHA
        R = sparse.diags(np.log((p / p.sum()) / (q / q.sum())).astype(np.float32))
        D[:, i] = LinearSVC(C=SVM_C).fit(Xfit @ R, pos.astype(int)).decision_function(Xapp @ R)
    return D

def cnb_scores(Xfit, yfit, Xapp):
    m = ComplementNB(alpha=0.3).fit(Xfit, yfit)
    assert list(m.classes_) == list(GROUPS)
    return m.predict_log_proba(Xapp).astype(np.float32)

## 4. Variables para el meta-modelo

### 4.1 G6: homofilia de red de la v3
Para cada `@cuenta` y cada `#hashtag` se calcula la distribución de edades de los tweets de entrenamiento que la contienen (*target encoding*; Micci-Barreca, 2001). Las distribuciones de las cuentas del tweet se suman y se suavizan hacia la distribución global con `prior=10`, para que una cuenta vista una sola vez no domine. Se basa en la **homofilia**: las personas se relacionan con quienes se les parecen (McPherson et al., 2001), y en Twitter eso permite inferir atributos a partir de la red (Al Zamal et al., 2012).

In [ ]:
def encode_entities(tok_fit, y_fit, tok_app, prefix, prior=10.0):
    """Codificación por objetivo: tok_* son listas de entidades por tweet (o textos si se usa token_pattern)."""
    if isinstance(prefix, tuple):                      # (patrón regex, prefijo): extrae entidades del texto
        pat, prefix = prefix
        cv = CountVectorizer(token_pattern=pat, lowercase=True, binary=True)
    else:
        cv = CountVectorizer(analyzer=lambda x: x, binary=True)
    Xf, Xp = cv.fit_transform(tok_fit), cv.transform(tok_app)
    Y = (np.asarray(y_fit)[:, None] == GROUPS[None, :]).astype(np.float32)
    C = np.asarray(Xf.T @ Y)                          # conteo de tweets por entidad y grupo de edad
    glob = Y.mean(0); S = np.asarray(Xp @ C); n = S.sum(1, keepdims=True)
    P = (S + prior * glob) / (n + prior)               # distribución suavizada hacia la global
    out = pd.DataFrame(P, columns=[f'{prefix}_p_{c}' for c in GROUPS])
    out[f'{prefix}_edad_media'] = P @ np.arange(len(GROUPS))
    out[f'{prefix}_soporte'] = np.log1p(n.ravel())
    out[f'{prefix}_n'] = np.asarray(Xp.sum(1)).ravel()
    return out.astype(np.float32)

def g6_homofilia(T_fit, y_fit, T_app):
    return pd.concat([encode_entities(T_fit, y_fit, T_app, (r'@\w+', 'cta')),
                      encode_entities(T_fit, y_fit, T_app, (r'#\w+', 'hsh'))], axis=1)

### 4.2 H5: homofilia ampliada *(nuevo en v4)*
La G6 mezcla todas las cuentas. Pero **no es lo mismo retuitear a alguien que responderle**: retuitear a un medio de noticias dice algo distinto que conversar con un amigo. H5 separa cinco tipos de relación y codifica cada uno por su cuenta:

| Relación | Ejemplo | Presente en |
|---|---|---|
| `rt_fuente` | la cuenta original de un RT: `RT @eltiempo: ...` | 36 % de los tweets |
| `ment_propia` | cuentas mencionadas o respondidas en tweets propios | 27 % |
| `ment_en_rt` | cuentas que aparecen dentro del texto retuiteado | 10 % |
| `hashtag` | hashtags | 20 % |
| `enlace` | el código del enlace `t.co/XXXX`: el mismo enlace compartido por varias personas | 39 % |

El **código del enlace** es la novedad más interesante: en el análisis de la v2, fragmentos de URL aparecían entre los términos más fuertes del modelo, señal de que los mismos enlaces circulan dentro de una misma comunidad de edad.

Juntas, G6 y H5 suman +0,004 sobre G6 sola. H5 sola no supera a G6, porque cada relación por separado tiene menos datos; se complementan.

In [ ]:
RT_SRC, MENT, HASH, URL = re.compile(r'^RT @(\w+)', re.I), re.compile(r'@(\w+)'), re.compile(r'#(\w+)'), re.compile(r't\.co/(\w+)')
KINDS = ['rt_fuente', 'ment_propia', 'ment_en_rt', 'hashtag', 'enlace']

def relations(texts):
    """Separa las entidades de cada tweet según el tipo de relación."""
    out = {k: [] for k in KINDS}
    for t in texts:
        m = RT_SRC.match(t); src = m.group(1).lower() if m else None
        ments = [x.lower() for x in MENT.findall(t)]
        out['rt_fuente'].append([src] if src else [])
        out['ment_en_rt'].append(ments[1:] if src else [])
        out['ment_propia'].append([] if src else ments)
        out['hashtag'].append([x.lower() for x in HASH.findall(t)])
        out['enlace'].append(URL.findall(t))
    return out

def h5_homofilia(R_fit, y_fit, R_app):
    return pd.concat([encode_entities(R_fit[k], y_fit, R_app[k], k) for k in KINDS], axis=1)

### 4.3 G0: estilo y legibilidad
Las variables de estilo de la v1 (longitud, pronombres, menciones, URL, RT, risas, mayúsculas, alargamientos) y los índices de legibilidad de Szigriszt-Pazos (1993) y Fernández Huerta (1959), con sílabas aproximadas por grupos vocálicos. En la v3 no entraron porque el selector usaba regresión logística; con gradient boosting suman +0,006, porque el meta-modelo las combina con las puntuaciones de NB-SVM (por ejemplo, "NB-SVM duda entre 18-24 y 25-34, pero el tweet es largo y formal").

In [ ]:
PRON = {'p1s': r'\b(?:yo|me|mi|mis|conmigo)\b', 'p2s_tu': r'\b(?:tu|tus|te|ti|contigo)\b', 'p2s_vos': r'\bvos\b',
        'p2s_ud': r'\b(?:usted|ud|uds|ustedes)\b', 'p1p': r'\b(?:nosotros|nosotras|nos|nuestro|nuestra|nuestros|nuestras)\b',
        'p3s': r'\b(?:el|ella|le|lo|la|su|sus)\b', 'p3p': r'\b(?:ellos|ellas|les|los)\b'}

def g0_estilo(texts):
    s = pd.Series(texts).reset_index(drop=True); low = s.str.lower(); w = low.str.findall(r'[a-z]+'); f = pd.DataFrame(index=s.index)
    f['len'] = s.str.len(); f['n_words'] = s.str.split().str.len().fillna(0)
    f['avg_wlen'] = w.apply(lambda x: np.mean([len(t) for t in x]) if x else 0); f['truncado'] = (f['len'] >= 135).astype(int)
    for k, p in PRON.items(): f[k] = low.str.count(p)
    f['n_hashtag'] = s.str.count(r'#\w+'); f['n_mention'] = s.str.count(r'@\w+'); f['n_url'] = s.str.count(r'https?://')
    f['rt'] = s.str.startswith('RT @').astype(int); f['respuesta'] = s.str.match(r'^@\w').astype(int)
    f['risa'] = low.str.contains(r'(?:ja){2,}|(?:je){2,}|jsjs|\bxd+\b').astype(int)
    f['alarg'] = low.str.contains(r'([a-z])\1{2,}').astype(int)
    f['mayus_ratio'] = s.apply(lambda x: sum(c.isupper() for c in x) / max(1, sum(c.isalpha() for c in x)))
    clean = s.str.replace(r'https?://\S+|@\w+|#\w+|^RT\b', ' ', regex=True); ws = clean.str.lower().str.findall(r'[a-z]+')
    syl = ws.apply(lambda x: sum(max(1, len(re.findall(r'[aeiouy]+', t))) for t in x)); nw = ws.str.len().clip(lower=1)
    nf = clean.str.count(r'[.!?]+').clip(lower=1)
    f['szigriszt'] = 206.835 - 62.3 * (syl / nw) - (nw / nf)
    f['fernandez_huerta'] = 206.84 - 0.60 * (syl / nw * 100) - 1.02 * (nf / nw * 100)
    f['pocas_palabras'] = (ws.str.len() < 3).astype(int)
    return f.astype(np.float32)

## 5. Construcción de todas las entradas del meta-modelo

`build_all(T_fit, y_fit, T_app)` produce dos diccionarios de bloques:
- **FA (ajuste):** todo lo que usa etiquetas (los tres modelos base, G6 y H5) se calcula **fuera de muestra** con `K_FOLDS` particiones. Cada tweet recibe puntuaciones de modelos que no vieron su etiqueta; así el meta-modelo aprende con predicciones tan "honestas" como las que verá en el test.
- **FB (aplicación):** se calcula con modelos ajustados sobre todo `T_fit`.

En validación, `T_fit` es el 80 % del train y `T_app` el 20 % reservado. En el modelo final, `T_fit` es todo el train y `T_app` el test.

In [ ]:
def build_all(T_fit, y_fit, T_app):
    T_fit, T_app = T_fit.reset_index(drop=True), T_app.reset_index(drop=True); y_fit = np.asarray(y_fit)
    folds = list(StratifiedKFold(K_FOLDS, shuffle=True, random_state=0).split(np.zeros(len(y_fit)), y_fit))
    FA, FB = {}, {}

    log('TF-IDF (palabras 1-3, char_wb 2-5, char 1-5)...')
    vecs = make_vectorizers()
    Xf = sparse.hstack([v.fit_transform(T_fit) for v in vecs]).tocsr()
    Xa = sparse.hstack([v.transform(T_app) for v in vecs]).tocsr()
    n_w = len(vecs[0].vocabulary_)
    log('  dimensiones:', [len(v.vocabulary_) for v in vecs])

    def oof(fn, Xf_, Xa_, name):
        O = np.zeros((len(y_fit), len(GROUPS)), np.float32)
        for k, (i, j) in enumerate(folds):
            O[j] = fn(Xf_[i], y_fit[i], Xf_[j])
        FA[name] = pd.DataFrame(O, columns=[f'{name}_{c}' for c in GROUPS])
        FB[name] = pd.DataFrame(fn(Xf_, y_fit, Xa_), columns=FA[name].columns)
        log(f'  {name} listo')

    log('Modelos base fuera de muestra...')
    oof(nbsvm_scores, Xf, Xa, 'NB')                                  # NB-SVM completo
    oof(cnb_scores, Xf, Xa, 'cnb')                                   # ComplementNB
    Xfw, Xaw = Xf[:, :n_w], Xa[:, :n_w]
    del Xf, Xa; gc.collect()
    oof(nbsvm_scores, Xfw, Xaw, 'nbword')                            # NB-SVM solo palabras
    del Xfw, Xaw; gc.collect()

    log('G6 y H5 (homofilia, fuera de muestra)...')
    R_fit, R_app = relations(T_fit), relations(T_app)
    sub = lambda R, idx: {k: [R[k][i] for i in idx] for k in KINDS}
    p6, p5 = [], []
    for i, j in folds:
        f6 = g6_homofilia(T_fit.iloc[i], y_fit[i], T_fit.iloc[j]); f6.index = j; p6.append(f6)
        f5 = h5_homofilia(sub(R_fit, i), y_fit[i], sub(R_fit, j)); f5.index = j; p5.append(f5)
    FA['G6'] = pd.concat(p6).sort_index().reset_index(drop=True); FB['G6'] = g6_homofilia(T_fit, y_fit, T_app)
    FA['H5'] = pd.concat(p5).sort_index().reset_index(drop=True); FB['H5'] = h5_homofilia(R_fit, y_fit, R_app)

    log('G0 estilo y legibilidad...')
    FA['G0'], FB['G0'] = g0_estilo(T_fit), g0_estilo(T_app)
    log('Entradas listas:', {k: v.shape[1] for k, v in FA.items()})
    return FA, FB

BLOCKS = ['NB', 'nbword', 'cnb', 'G6', 'H5', 'G0']
def design(F, blocks=BLOCKS): return pd.concat([F[b] for b in blocks], axis=1).values.astype(np.float32)

## 6. Meta-modelo y pesos por clase

**Meta-modelo:** `HistGradientBoostingClassifier` con hojas de hasta 63 nodos y un mínimo de 50 tweets por hoja.

**Pesos por clase** *(nuevo en v4)*: el F1 macro premia por igual a las seis clases, pero el meta-modelo maximiza la probabilidad, no el F1. Se multiplica la probabilidad de cada clase por un peso y se elige la clase con mayor producto. Los pesos se buscan por ascenso coordenado (cada peso entre 0,6 y 1,6, tres pasadas) sobre predicciones **fuera de muestra del propio meta-modelo** dentro del conjunto de ajuste, nunca sobre la validación ni el test. Típicamente bajan el peso de 13-17 y 65-xx, que el modelo tiende a sobrepredecir, y suben el de 25-34.

In [ ]:
def fit_meta(A, y):
    sc = StandardScaler().fit(A)
    return sc, HistGradientBoostingClassifier(**HGB_PARAMS).fit(sc.transform(A), y)

def tune_class_weights(A, y):
    """Pesos por clase que maximizan el F1 macro sobre predicciones fuera de muestra del meta-modelo."""
    y = np.asarray(y); O = np.zeros((len(y), len(GROUPS)))
    for i, j in StratifiedKFold(3, shuffle=True, random_state=7).split(A, y):
        sc, m = fit_meta(A[i], y[i]); O[j] = m.predict_proba(sc.transform(A[j]))
    w = np.ones(len(GROUPS))
    for _ in range(3):
        for c in range(len(GROUPS)):
            best_s, best_v = f1_score(y, GROUPS[(O * w).argmax(1)], average='macro'), w[c]
            for v in np.linspace(0.6, 1.6, 21):
                w2 = w.copy(); w2[c] = v; s = f1_score(y, GROUPS[(O * w2).argmax(1)], average='macro')
                if s > best_s: best_s, best_v = s, v
            w[c] = best_v
    log('  F1 macro fuera de muestra del meta-modelo: sin pesos',
        round(f1_score(y, GROUPS[O.argmax(1)], average='macro'), 4), '| con pesos', round(best_s, 4))
    return w

## 7. Validación 80/20

Se repite todo con el 80 % del train como ajuste y el 20 % como validación, que no interviene en ninguna decisión. Se reportan:
1. **Ablación incremental:** cuánto aporta cada bloque al agregarlo en orden (equivale a un selector hacia adelante con el meta-modelo final).
2. **Métricas por clase y matriz de confusión** del modelo completo con pesos.
3. **Importancia por permutación de cada bloque:** cuánto cae el F1 si se desordena.

**Resultados de referencia** obtenidos con el train completo:

| Entradas | F1 macro |
|---|---|
| NB-SVM + G6 (equivalente v3) | 0,607 |
| + H5 | 0,611 |
| + nbword + cnb | 0,613 |
| + G0 | 0,618 |
| + pesos por clase | 0,619 |

Con muestras pequeñas (`SAMPLE_FRAC`) los números serán menores y más ruidosos.

In [ ]:
if VALIDAR:
    a, b = train_test_split(np.arange(len(train)), test_size=0.2, stratify=train.age_range, random_state=SEED)
    ya, yb = train.age_range.values[a], train.age_range.values[b]
    FA, FB = build_all(train.text.iloc[a], ya, train.text.iloc[b])

In [ ]:
if VALIDAR:
    ablation = [('NB-SVM + G6 (equivalente v3)', ['NB', 'G6']), ('+ H5 homofilia ampliada', ['NB', 'G6', 'H5']),
                ('+ nbword + cnb', ['NB', 'G6', 'H5', 'nbword', 'cnb']), ('+ G0 estilo (modelo v4)', BLOCKS)]
    rows = []
    for name, blocks in ablation:
        sc, m = fit_meta(design(FA, blocks), ya)
        P = m.predict_proba(sc.transform(design(FB, blocks)))
        rows.append((name, f1_score(yb, GROUPS[P.argmax(1)], average='macro')))
        log(f'  {name:32s} F1 = {rows[-1][1]:.4f}')
    w_val = tune_class_weights(design(FA), ya)
    pred_b = GROUPS[(P * w_val).argmax(1)]
    rows.append(('+ pesos por clase', f1_score(yb, pred_b, average='macro')))
    display(pd.DataFrame(rows, columns=['entradas del meta-modelo', 'F1 macro']).round(4))
    print('Pesos por clase:', dict(zip(GROUPS.tolist(), w_val.round(2).tolist())))

In [ ]:
if VALIDAR:
    print(classification_report(yb, pred_b, labels=GROUPS, digits=3))
    ConfusionMatrixDisplay.from_predictions(yb, pred_b, labels=GROUPS, normalize='true', values_format='.2f', cmap='Blues')
    plt.title('Matriz de confusión normalizada (validación, v4)'); plt.show()

**Importancia por permutación:** se desordena cada bloque en la validación (tres repeticiones) y se mide la caída del F1. Una caída grande indica que el meta-modelo depende mucho de ese bloque. Como los bloques están correlacionados (por ejemplo, NB-SVM y nbword), un bloque puede parecer poco importante porque otro lo reemplaza.

In [ ]:
if VALIDAR:
    Bs = sc.transform(design(FB)); base = f1_score(yb, GROUPS[(m.predict_proba(Bs) * w_val).argmax(1)], average='macro')
    cols = [c for blk in BLOCKS for c in FB[blk].columns]; imp = {}
    for blk in BLOCKS:
        idx = [cols.index(c) for c in FB[blk].columns]; drops = []
        for r in range(3):
            Pm = Bs.copy(); Pm[:, idx] = Pm[np.random.RandomState(r).permutation(len(Pm))][:, idx]
            drops.append(base - f1_score(yb, GROUPS[(m.predict_proba(Pm) * w_val).argmax(1)], average='macro'))
        imp[blk] = np.mean(drops)
    pd.Series(imp).sort_values().plot.barh(title='Caída del F1 macro al permutar cada bloque'); plt.show()
    del FA, FB; gc.collect()

## 8. Modelo final con todo el train

Se recalculan todas las entradas con el 100 % del train (fuera de muestra) y con el test, se ajustan los pesos por clase sobre el train y se entrena el meta-modelo. Luego se aplican las dos reglas validadas en la v2:
1. **Duplicados exactos:** un tweet de test idéntico a uno de train con etiqueta unánime recibe esa etiqueta.
2. **Vacíos y `#¿NOMBRE?`:** reciben la clase más frecuente de esos casos en el train.

In [ ]:
FA, FB = build_all(train.text, train.age_range.values, test.text)
log('Pesos por clase sobre todo el train...')
w = tune_class_weights(design(FA), train.age_range.values)
print('Pesos por clase:', dict(zip(GROUPS.tolist(), w.round(2).tolist())))
sc, meta = fit_meta(design(FA), train.age_range.values)
test['age_range'] = GROUPS[(meta.predict_proba(sc.transform(design(FB))) * w).argmax(1)]
log('Predicciones del meta-modelo listas')

In [ ]:
lab = train.groupby('text').age_range.agg(['nunique', 'first']); lookup = lab.loc[lab['nunique'] == 1, 'first']
hit = test.text.isin(lookup.index) & (test.text != '')
test.loc[hit, 'age_range'] = test.loc[hit, 'text'].map(lookup)
jt = is_junk(train_raw.text)
default = train_raw.age_range[jt].mode().iloc[0] if jt.any() else train.age_range.mode().iloc[0]
test.loc[is_junk(test.text), 'age_range'] = default
log(f'Duplicados reemplazados: {int(hit.sum())} | vacíos: {int(is_junk(test.text).sum())} -> {default}')

sp = os.path.join(DATA_DIR, 'age_sample_submission.csv')
cols = (pd.read_csv(sp, nrows=0, sep=None, engine='python', encoding='utf-8-sig').columns.tolist()
        if os.path.exists(sp) else ['id', 'age_range'])
sub = pd.DataFrame({cols[0]: test['id'].astype(int), cols[1]: test['age_range']})
assert sub[cols[1]].isin(GROUPS).all() and len(sub) == len(test)
sub.to_csv(OUT_PATH, index=False)
log('Guardado:', os.path.abspath(OUT_PATH), sub.shape)
sub[cols[1]].value_counts(normalize=True).reindex(GROUPS).round(3)

## 9. Conclusiones
- **La relación con otras cuentas es la señal más rentable** sin modelos preentrenados. Separar el tipo de relación (RT, mención propia, mención dentro de un RT, hashtag, enlace) y combinarlo con la homofilia de la v3 suma +0,004.
- **La combinación no lineal cambia qué variables sirven:** las de estilo y legibilidad, descartadas por el selector lineal de la v3, aportan +0,006 cuando el meta-modelo es gradient boosting.
- **La diversidad debe ser barata:** NB-SVM de palabras y ComplementNB aportan lo mismo que modelos mucho más costosos (NB-SVM de caracteres, clasificadores ordinales), que se descartaron.
- **Ajustar los pesos por clase** alinea la decisión con la métrica (F1 macro) sin tocar el test.
- **Próximo notebook:** una red tipo fastText entrenada desde cero como modelo base adicional.

## Referencias (APA 7)

Al Zamal, F., Liu, W., & Ruths, D. (2012). Homophily and latent attribute inference: Inferring latent attributes of Twitter users from neighbors. *Proceedings of the International AAAI Conference on Web and Social Media, 6*(1), 387–390.

Fernández Huerta, J. (1959). Medidas sencillas de lecturabilidad. *Consigna, 214*, 29–32.

Frank, E., & Hall, M. (2001). A simple approach to ordinal classification. En *Machine Learning: ECML 2001* (pp. 145–156). Springer.

Friedman, J. H. (2001). Greedy function approximation: A gradient boosting machine. *The Annals of Statistics, 29*(5), 1189–1232.

McPherson, M., Smith-Lovin, L., & Cook, J. M. (2001). Birds of a feather: Homophily in social networks. *Annual Review of Sociology, 27*, 415–444.

Micci-Barreca, D. (2001). A preprocessing scheme for high-cardinality categorical attributes in classification and prediction problems. *ACM SIGKDD Explorations Newsletter, 3*(1), 27–32.

Moreno-Sandoval, L. G., Pomares-Quimbaya, A., & Alvarado-Valencia, J. A. (2021). Celebrity profiling through linguistic analysis of digital social networks. *Computational Social Networks, 8*, Artículo 16.

Rennie, J. D. M., Shih, L., Teevan, J., & Karger, D. R. (2003). Tackling the poor assumptions of naive Bayes text classifiers. En *Proceedings of the 20th International Conference on Machine Learning* (pp. 616–623).

Szigriszt Pazos, F. (1993). *Sistemas predictivos de legibilidad del mensaje escrito: Fórmula de perspicuidad* [Tesis doctoral, Universidad Complutense de Madrid].

Wang, S., & Manning, C. D. (2012). Baselines and bigrams: Simple, good sentiment and topic classification. En *Proceedings of the 50th Annual Meeting of the Association for Computational Linguistics* (Vol. 2, pp. 90–94).

Wolpert, D. H. (1992). Stacked generalization. *Neural Networks, 5*(2), 241–259.